# Places - Cleaned Analysis

Analysis of places in the individual biographies.

This notebook collects all the places mentioned in the individual biographies and uses Wikidata to extract associated information about them, namely current names, coordinates, and administrative information (present day).

Relevant to this notebook is the list of attribute types that receive place names as values:

- 'nascimento'
- 'jesuita-entrada'
- 'partida'
- 'chegada'
- 'estadia'
- 'estadia-x'
- 'jesuita-votos-local'
- 'jesuita-ordenacao-padre'
- 'morte'

## Configuration

In [ ]:
# Configuration parameters
PLACE_ATTRIBUTES = [
    'nascimento', 'jesuita-entrada', 'partida', 'chegada', 
    'estadia', 'estadia-x', 'jesuita-votos-local', 
    'jesuita-ordenacao-padre', 'morte'
]

# Columns to display in results
DISPLAY_COLS = [
    'place', 'place.wikidata_id', 'place.date_inferred.formatted', 
    'place.date_inferred', 'groupname', 'name', 'place.type', 
    'nascimento', 'nascimento.date', 'place.type', 'jesuita-entrada', 
    'jesuita-entrada.date', 'place.line', 'place.extra_info'
]

# Default number of rows to display
SHOW_ONLY = 10

# Place of entry for specific analysis
PLACE_OF_ENTRY = 'Coimbra'

## Imports

In [2]:
import pandas as pd
import re
import os
from timelink.notebooks import TimelinkNotebook
from timelink.pandas import entities_with_attribute
from timelink.kleio.utilities import format_timelink_date

# Set pandas display options
pd.set_option('display.max_rows', 150)

## Helper Functions

In [ ]:
def get_place_attributes():
    """Define attribute types that have locations as values."""
    return PLACE_ATTRIBUTES

def load_places_data(db_connection):
    """Load place data from the database and perform initial preprocessing."""
    print("Loading places data...")
    
    # Load the data
    places_df = entities_with_attribute(
        entity_type='person',
        show_elements=['name', 'groupname', 'the_line', 'the_order', 'extra_info'],
        the_type=[
            'jesuita-entrada', 'partida', 'chegada', 'estadia', 'estadia-x',
            'jesuita-votos-local', 'jesuita-ordenacao-padre', 'morte'
        ],
        column_name='place',
        more_attributes=['jesuita-entrada', 'nascimento'],
        db=db_connection,
    )
    
    # Preprocess: fill NaN values and remove unknown places
    places_df.place = places_df.place.fillna('?')
    places_df = places_df[places_df.place != '?']
    
    return places_df

def infer_dates(df):
    """Infer missing dates based on previous entries for each person."""
    df = df.copy()
    df['id_col'] = df.index
    
    # Create a copy of the date column to replace '0' values with NaN
    df['place.date_inferred'] = df['place.date'].replace('0', pd.NA)
    
    # Sort by id and line to have the missing date values filled with the previous date
    df = df.sort_values(by=['id_col', 'place.line'])
    
    # Use ffill to fill the missing values with the previous value
    df['place.date_inferred'] = df.groupby(['id_col'])['place.date_inferred'].ffill()
    
    # Create a column to flag the inferred dates
    df['place.date_is_inferred'] = df['place.date_inferred'] != df['place.date']
    
    # Reset index temporarily
    df.reset_index(inplace=True)
    
    # If data_is_inferred, set date_inferred to date_inferred + ">"
    df.loc[df['place.date_is_inferred'], 'place.date_inferred'] = df['place.date_inferred'] + '>'
    
    # Set missing value to ''
    df['place.date_inferred'] = df['place.date_inferred'].fillna('')
    
    # Format dates
    df['place.date_inferred.formatted'] = df['place.date_inferred'].apply(format_timelink_date)
    
    # Restore index
    df.index = df['id_col']
    
    return df

def infer_dates_loop(df):
    """Infer missing dates using an explicit loop logic."""
    df = df.copy()
    df['id_col'] = df.index
    
    # Initialize the inferred column, treating '0' as missing
    df['place.date_inferred'] = df['place.date'].replace('0', pd.NA)
    
    # strict sorting is required for "previous line" logic to work
    df = df.sort_values(by=['id_col', 'place.line'])
    
    # Variables to track state inside the loop
    last_id = None
    last_valid_date = pd.NA
    
    # Get column integer locations for positional access (avoiding index ambiguity)
    id_col_idx = df.columns.get_loc('id_col')
    date_col_idx = df.columns.get_loc('place.date_inferred')
    date_original_col_idx = df.columns.get_loc('place.date')
    place_col_idx = df.columns.get_loc('place')
    place_type_col_idx = df.columns.get_loc('place.type')

    # Iterate through every row by number
    for i in range(len(df)):
        current_id = df.iat[i, id_col_idx]
        current_date = df.iat[i, date_col_idx]
        original_date = df.iat[i, date_original_col_idx]
        current_place = df.iat[i, place_col_idx]
        current_place_type = df.iat[i, place_type_col_idx]
        
        # If we encounter a new person, reset the tracker
        if current_id != last_id:
            last_id = current_id
            last_valid_date = pd.NA
            
        if original_date == '0':
            # Missing date, we may need to infer
            print('Missing date')
            print(f"Person ID: {current_id}, Place: {current_place} ({current_place_type})")
            pass
        # If current date exists, update our "memory"
        if pd.notna(current_date):
            last_valid_date = current_date
        # If current date is missing but we have a date associated with this person
        elif pd.notna(last_valid_date):
            df.iat[i, date_col_idx] = last_valid_date

    # --- Formatting logic (same as original) ---
    
    # Flag inferred dates (comparing result vs original '0' or empty data)
    df['place.date_is_inferred'] = df['place.date_inferred'] != df['place.date']
    
    # Reset index temporarily for vectorized string operations
    df.reset_index(inplace=True)
    
    # Append marker '>' to inferred dates
    mask_inferred = df['place.date_is_inferred'] & df['place.date_inferred'].notna()
    df.loc[mask_inferred, 'place.date_inferred'] = '>' + df.loc[mask_inferred, 'place.date_inferred'].astype(str)
    
    # Clean up NaNs for clean display
    df['place.date_inferred'] = df['place.date_inferred'].fillna('')
    
    # Format using Timelink utility
    df['place.date_inferred.formatted'] = df['place.date_inferred'].apply(format_timelink_date)
    
    # Restore original index structure
    df.index = df['id_col']
    
    return df

def extract_wikidata_id(extra_info_dict):
    """
    Extract Wikidata ID from place.extra_info JSON string.
    Returns the Wikidata ID (e.g., 'Q6625') or NaN if not present.
    """
    if pd.isna(extra_info_dict):
        return pd.NA
    
    try:
        comment = extra_info_dict.get('the_value', {}).get('comment', '')
        if comment and 'wikidata:' in comment:
            # Extract Wikidata ID using regex pattern (Q followed by digits)
            match = re.search(r'wikidata:(Q\d+)', comment)
            if match:
                return match.group(1)
        return pd.NA
    except (ValueError, SyntaxError, TypeError):
        return pd.NA

def extract_wikidata_ids(df):
    """Extract Wikidata IDs from extra info column."""
    df = df.copy()
    df['place.wikidata_id'] = df['place.extra_info'].apply(extract_wikidata_id)
    return df

def export_results(df, cols=DISPLAY_COLS):
    """Export results to various Excel files."""
    print("Exporting results to Excel files...")
    
    # Export full data with Wikidata IDs
    print("Exporting full data with Wikidata IDs...")
    df[cols].sort_values(by=['place', 'place.date_inferred']).to_excel(
        "../inferences/jesuita_entrada_places_of_stay.xlsx", 
        sheet_name='wikidata', 
        index=True
    )
    
    # Export data without Wikidata IDs for unbiased analysis
    print("Exporting data without Wikidata IDs...")
    df[['place', 'place.date_inferred.formatted', 'name', 'place.type', 'place.original']].sort_values(
        by=['place', 'place.date_inferred.formatted']
    ).to_excel(
        "../inferences/locations_names_places_dates_only.xlsx", 
        sheet_name='wikidata', 
        index=True
    )
    
    # Export unlocated places
    print("Exporting unlocated places...")
    df[df['place.wikidata_id'].isna()][cols].sort_values(
        by=['place', 'place.date_inferred']
    ).to_excel(
        "../inferences/locations_names_no_wikidata.xlsx", 
        sheet_name='no_wikidata', 
        index=True
    )
    
    # Export Wikidata IDs for later processing
    print("Exporting Wikidata IDs...")
    wikidata_ids = df[df['place.wikidata_id'].notna()]['place.wikidata_id'].drop_duplicates()
    if len(wikidata_ids) > 0:
        wikidata_ids.to_csv(
            "../inferences/wikidata-references/locations_names_wikidata.csv", 
            index=False
        )
    
    print(f"Exported {len(df)} records and {len(wikidata_ids)} Wikidata IDs")

def analyze_specific_place_of_entry(df, place_of_entry=PLACE_OF_ENTRY, cols=DISPLAY_COLS):
    """Analyze a specific location (default: Coimbra)."""
    from timelink.pandas import entities_with_attribute
    
    print(f"Analyzing missionaries with place of entry = {place_of_entry}...")
    
    # Get the IDs of the places of entry
    missionaries_from_place = entities_with_attribute(
        entity_type='person',
        show_elements=['name', 'groupname', 'the_line', 'the_order', 'extra_info'],
        the_type=['jesuita-entrada'],
        the_value=place_of_entry,
        db=tlnb.db,
    )
    
    print(f"Number of missionaries from {place_of_entry}: {len(missionaries_from_place)}")
    place_of_entry_ids = missionaries_from_place.index
    places_of_stay_specific = df.loc[place_of_entry_ids]
    
    # Export unlocated places for this specific location
    unlocated = places_of_stay_specific[places_of_stay_specific['place.wikidata_id'].isna()]
    if len(unlocated) > 0:
        unlocated[cols].sort_values(by=['place', 'place.date_inferred']).to_excel(
            f"../inferences/jesuita_entrada_{place_of_entry}_places_of_stay_no_wikidata.xlsx",
            sheet_name='no_wikidata',
            index=True
        )
        print(f"Exported {len(unlocated)} unlocated places from {place_of_entry}")
    
    return places_of_stay_specific

def generate_frequency_report(df):
    """Generate frequency report of places."""
    print("Generating frequency report...")
    
    # Use pandas groupby and specify unique value count for id
    df_totals = df.groupby('place').agg({
        'id': 'nunique',
        'place.date_inferred': 'min',
        'place.date_inferred': 'max'
    })
    
    # Export totals to Excel
    df_totals.sort_values('id', ascending=False).to_excel(
        "../inferences/locations_totals.xlsx", 
        sheet_name='totals', 
        index=True
    )
    
    print(f"Generated frequency report for {len(df_totals)} locations")
    return df_totals

def show_who_was_at_place(df, place="Coimbra", show_only=100):
    """Show who was at a specific place."""
    cols = ['place', 'name', 'place.type', 'place.date_inferred', 'place.obs']
    
    result = df[df.place == place][cols].sort_values(by=['place.date_inferred'])
    print(f"Found {len(result)} records for {place}")
    
    if len(result) > 0:
        display(result.head(show_only))
    
    return result

def export_to_markdown(df, place="Coimbra", place_of_entry=PLACE_OF_ENTRY):
    """Export specific place data to markdown files."""
    print(f"Exporting {place} data to markdown files...")
    
    # Create directory if it doesn't exist
    directory = f"../inferences/{place_of_entry}/markdown"
    if not os.path.exists(directory):
        os.makedirs(directory)
    
    # Get the records for this place
    place_records = df[df.place == place][['name']].sort_values('name')
    
    print(f"Exporting {len(place_records)} records to markdown...")
    
    for id, name in place_records.itertuples(index=True):
        with tlnb.db.session() as session:
            p = tlnb.db.get_person(id, session=session)
            session.add(p)
            pk: str = p.to_kleio()
            pk = f"""## {{name}}
^[In Edit mode, select title (including the ##), right click and "Extract selection"]

---
### {{name}} @ timelink
```
{{pk}}
```

---
Select title, right click on selected title, choose extract note
"""
            fname = f"{directory}/{id}.md"
            with open(fname, 'w', encoding='utf-8') as f:
                f.write(pk)
    
    print(f"Successfully exported {len(place_records)} markdown files to {{directory}}")

## Main Execution

In [10]:
# Initialize notebook
tlnb = TimelinkNotebook()
tlnb.print_info()

Timelink version: 1.1.30
Project name: dehergne
Project home: /Users/jrc/mhk-home/sources/dehergne
Database type: sqlite
Database name: dehergne
Kleio image: timelinkserver/kleio-server
Kleio server token: u8fTc...
Kleio server URL: http://127.0.0.1:8089
Kleio server home: /Users/jrc/mhk-home/sources/dehergne
Kleio server container: upbeat_engelbart
Kleio version requested: latest
Kleio server version: 12.9.588 (2025-06-06 16:04:22)
SQLite directory: /Users/jrc/mhk-home/sources/dehergne/database/sqlite
Database version: 6ccf1ef385a6
Call print_info(show_token=True) to show the Kleio Server token
Call print_info(show_password=True) to show the Postgres password
TimelinkNotebook(project_name=dehergne, project_home=/Users/jrc/mhk-home/sources/dehergne, db_type=sqlite, db_name=dehergne, kleio_image=timelinkserver/kleio-server, kleio_version=latest, postgres_image=postgres, postgres_version=latest)


In [15]:
# Load and process the data
places_of_stay = load_places_data(tlnb.db)
places_of_stay = infer_dates_loop(places_of_stay)
places_of_stay = extract_wikidata_ids(places_of_stay)

print(f"Processed data shape: {places_of_stay.shape}")
places_of_stay.info()

Loading places data...
Missing date
Person ID: deh-adam-algenler, Place: Brasil (estadia)
Missing date
Person ID: deh-andre-pereira, Place: Macau (estadia)
Missing date
Person ID: deh-antoine-chomel-ref2, Place: China (estadia)
Missing date
Person ID: deh-antoine-chomel-ref3, Place: Toulouse (estadia)
Missing date
Person ID: deh-antonio-de-almeida, Place: Macau (estadia)
Missing date
Person ID: deh-antonio-de-andrade, Place: Himalaias (estadia)
Missing date
Person ID: deh-antonio-de-andrade, Place: Tsaparang, Tibete (estadia)
Missing date
Person ID: deh-antonio-de-melo, Place: Sião (estadia)
Missing date
Person ID: deh-antonio-de-melo, Place: Tonquim (estadia)
Missing date
Person ID: deh-antonio-de-torres, Place: Tchampa (estadia)
Missing date
Person ID: deh-antonio-de-torres, Place: Macau (estadia)
Missing date
Person ID: deh-antonio-diaz, Place: Índia (estadia)
Missing date
Person ID: deh-antonio-ferreira, Place: Macau (estadia)
Missing date
Person ID: deh-antonio-francisco-cardim, P

In [16]:
places_of_stay[['id','place.date_inferred','place.date','place.line']].sample(SHOW_ONLY)

,id,place.date_inferred,place.date,place.line
id_col,,,,
deh-balthasar-miller,deh-balthasar-miller,17270000,17270000,1599
deh-jean-forget,deh-jean-forget,16431227,16431227,1406
deh-kaspar-castner,deh-kaspar-castner,17000815,17000815,1024
deh-francesco-folleri,deh-francesco-folleri,17210906,17210906,1149
deh-dominique-lou-sseu-mei,deh-dominique-lou-sseu-mei,17040619,17040619,1625
deh-louis-porquet,deh-louis-porquet,16861023,16861023,1718
deh-jan-baptista-chrzciciel-bakowski,deh-jan-baptista-chrzciciel-bakowski,1726,1726,120
deh-nicolas-trigault-junior,deh-nicolas-trigault-junior,16291002,16291002,1051
deh-joseph-labbe,deh-joseph-labbe,1731,1731,48


In [18]:
# Select columns relevant to date inference
inference_cols = ['name', 'place', 'place.date', 'place.date_inferred', 'place.date_is_inferred']

# Show only the rows where dates were actually inferred
print("\nSample of rows where dates were inferred:")
display(places_of_stay[places_of_stay['place.date_is_inferred']][inference_cols].sample(10))


Sample of rows where dates were inferred:


,name,place,place.date,place.date_inferred,place.date_is_inferred
id_col,,,,,
deh-leon-pascal-baron,Léon-Pascal Baron,Pequim,0,>17690000,True
deh-jacques-motel,Jacques Motel,Cantão,0,>16630000,True
deh-michael-pierre-boym,Michael-Pierre Boym,Pérsia,0,>16510500,True
deh-sebastiao-vieira,Sebastião Vieira,Macau,0,>16271129,True
deh-jean-baborier,Jean Baborier,"Hangchow, Tchö-kiang",0,>17240000,True
deh-ferdinando-bonaventura-moggi-ref1,Ehrenbert Xaver Fridelli,Pequim,0,,True
deh-philipp-sibin,Philipp Sibin,Sião,0,>1722,True
deh-gerard-arnould-rijckewaert,Gerard Arnould Rijckewaert,Pérsia,0,>1696,True
deh-jean-baptiste-joseph-de-grammont,Jean-Baptiste-Joseph de Grammont,Pequim (corte),0,>17690725,True


In [19]:
# Show sample results
sample_results = places_of_stay[places_of_stay.groupname == 'n'][DISPLAY_COLS].sort_values(
    by=['place', 'place.date_inferred', 'name']
).sample(SHOW_ONLY)

display(sample_results)

,place,place.wikidata_id,place.date_inferred.formatted,place.date_inferred,groupname,name,place.type,nascimento,nascimento.date,place.type,jesuita-entrada,jesuita-entrada.date,place.line,place.extra_info
id_col,,,,,,,,,,,,,,
deh-carlo-amiani,Macau,Q14773,1723-02-22,17230222,n,Carlo Amiani,morte,"Fano, Itália",16610106,morte,Roma,16790512,963,"{'the_value': {'kleio_element_name': 'valor', ..."
deh-domenico-fuciti,Vannes,Q6593,1654-10,16541000,n,Domenico Fuciti,estadia,Nápoles,1625,estadia,Nápoles,16381110,1719,"{'the_value': {'kleio_element_name': 'valor', ..."
deh-giacomo-filippo-simonelli,"Tsinan fou, Shantung",Q170247,1722-11-26,17221126,n,Giacomo Filippo Simonelli,partida,Macerata,16800625,partida,?,17031031,1613,"{'id': {'kleio_element_name': 'id', 'kleio_ele..."
deh-ludovico-antonio-adorno,Génova,Q1449,1680-12-02,16801202,n,Lodovico António Adorno,jesuita-entrada,"Génova, Itália",16550828,jesuita-entrada,Génova,16801202,109,"{'the_value': {'kleio_element_name': 'valor', ..."
deh-joao-de-sequeira,Tonquim,Q622778,1694,1694,n,João de Sequeira,estadia,Portalegre,1663,estadia,Évora,16801024,920,"{'id': {'kleio_element_name': 'id', 'kleio_ele..."
deh-gil-martinez-de-la-mata,Lisboa,Q597,1596-04-10,15960410,n,Gil Martínez de la Mata,estadia,"Logroño, diocese de Calahorra",15480000,estadia,?,15660312,965,"{'the_type': {'kleio_element_name': 'tipo', 'k..."
deh-johann-balthasar-staubach,Goa,Q1171,1690-11-02,16901102,n,Johann Balthasar Staubach,chegada,Fulda,16590401,chegada,Roma,16760826,2302,"{'id': {'kleio_element_name': 'id', 'kleio_ele..."
deh-joao-cabral,Ceilão,Q854,>1627,>1627,n,João Cabral,estadia,Celorico da Beira,1598,estadia,?,16150613,74,"{'the_type': {'kleio_element_name': 'tipo', 'k..."
deh-cristovao-da-costa,Malaca,Q61089,1561,1561,n,Cristóvão da Costa,estadia,?,1529,estadia,Coimbra,15500103,2213,"{'the_type': {'kleio_element_name': 'tipo', 'k..."


In [20]:
# Export all results
export_results(places_of_stay)

Exporting results to Excel files...
Exporting full data with Wikidata IDs...
Exporting data without Wikidata IDs...
Exporting unlocated places...
Exporting Wikidata IDs...
Exported 6297 records and 542 Wikidata IDs


In [21]:
# Analyze specific location (e.g., Coimbra)
place = "Coimbra"
places_of_stay_specific = analyze_specific_place_of_entry(places_of_stay, place_of_entry=place)

Analyzing missionaries with place of entry = Coimbra...
Number of missionaries from Coimbra: 63
Exported 2 unlocated places from Coimbra


In [22]:
places_of_stay_specific.info()

<class 'pandas.core.frame.DataFrame'>
Index: 537 entries, deh-pedro-de-alcacova to deh-inacio-pires
Data columns (total 40 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   id                             537 non-null    object
 1   name                           537 non-null    object
 2   groupname                      537 non-null    object
 3   the_line                       537 non-null    int64 
 4   the_order                      537 non-null    int64 
 5   extra_info                     537 non-null    object
 6   place.attr_id                  537 non-null    object
 7   place.type                     537 non-null    object
 8   place                          537 non-null    object
 9   place.date                     537 non-null    object
 10  place.line                     537 non-null    int64 
 11  place.level                    537 non-null    int64 
 12  place.groupname                537 n

Show places mentioned in biographies, along with their cleaned information from Wikidata.

In [23]:
# Generate frequency report
frequency_report = generate_frequency_report(places_of_stay_specific)
display(frequency_report.sort_values('id', ascending=False).head(20))

Generating frequency report...
Generated frequency report for 136 locations


,id,place.date_inferred
place,,
Coimbra,63,<16230324
Macau,42,><16230324
Goa,25,>16940718
Pequim,14,>16581112
Cantão,11,17210524
Japão,11,16320712
China,9,1742
Hangchow,9,>1636
Índia,7,>16750213


In [24]:
# Generate frequency report
frequency_report = generate_frequency_report(places_of_stay)
display(frequency_report.sort_values('id', ascending=False).head(20))

Generating frequency report...
Generated frequency report for 1099 locations


,id,place.date_inferred
place,,
Macau,449,><16230324
Pequim,250,>17690000
Goa,209,>17630322
Cantão,197,>17271019
China,180,>1750
Lisboa,130,>17471123
Roma,110,>17080114
Coimbra,78,>16060000
Paris,65,>17700110


In [25]:
# Show who was at a specific place (e.g., Coimbra)
coimbra_visitors = show_who_was_at_place(places_of_stay, place="Coimbra")

Found 89 records for Coimbra


,place,name,place.type,place.date_inferred,place.obs
id_col,,,,,
deh-pedro-de-alcacova,Coimbra,Pedro de Alcáçova,jesuita-entrada,15420000,saiu a primeira vez
deh-pedro-de-alcacova,Coimbra,Pedro de Alcáçova,jesuita-entrada,15420000,saiu a primeira vez
deh-belchior-nunes-barreto,Coimbra,Belchior Nunes Barreto,jesuita-entrada,15430311,
deh-belchior-miguel-carneiro-leitao,Coimbra,Belchior Miguel Carneiro Leitão,jesuita-entrada,15430425,
deh-francisco-perez,Coimbra,Francisco Pérez,jesuita-entrada,15440125,
deh-tiburcio-de-quadros,Coimbra,Tibúrcio de Quadros,jesuita-entrada,15440418,
deh-goncalo-alvares,Coimbra,Gonçalo Álvares,jesuita-entrada,15490101,
deh-cristovao-da-costa,Coimbra,Cristóvão da Costa,jesuita-entrada,15500103,
deh-joao-rodrigues-de-sainan,Coimbra,João Rodrigues de Sainan,jesuita-entrada,1555,


In [27]:
# Export to markdown
export_to_markdown(places_of_stay, place="Coimbra")

<class 'pandas.core.frame.DataFrame'>
Index: 6297 entries, deh-abraham-le-royer to simao-rodrigues-ref1
Data columns (total 40 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   id                             6297 non-null   object
 1   name                           6297 non-null   object
 2   groupname                      6297 non-null   object
 3   the_line                       6297 non-null   int64 
 4   the_order                      6297 non-null   int64 
 5   extra_info                     6297 non-null   object
 6   place.attr_id                  6297 non-null   object
 7   place.type                     6297 non-null   object
 8   place                          6297 non-null   object
 9   place.date                     6297 non-null   object
 10  place.line                     6297 non-null   int64 
 11  place.level                    6297 non-null   int64 
 12  place.groupname                6

KeyboardInterrupt: 

## Summary

Analysis complete! This cleaned notebook performs the same functions as the original but with:

1. Better organization into logical sections
2. Reusable functions to avoid code duplication
3. Centralized configuration parameters
4. Improved readability and maintainability
5. Consistent error handling and logging

All exports and analysis results are saved to the ../inferences directory.